# Uber Ride Analysis Project

**Name:** Jayakumar P  
**Dataset:** `uberdrive.csv` (My Uber Drives dataset)

This notebook follows the supplied project code and README: data loading, cleaning, exploratory analysis, visualizations, and a simple Random Forest demand model.


## Import Libraries

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score


## Step 1 — Load the Data

In [ ]:
# Load the data
data = pd.read_csv("uberdrive.csv")

print(data.head())
print(data.shape)
print(data.info())


## Step 2 — Clean the Data

In [ ]:
# Clean column names
data.columns = [col.replace("*", "") for col in data.columns]

# Remove the Totals row
data = data[data["START_DATE"] != "Totals"]

# Convert mixed date formats
data["START_DATE"] = pd.to_datetime(data["START_DATE"], format="mixed")
data["END_DATE"] = pd.to_datetime(data["END_DATE"], format="mixed")

# Remove rows with missing miles
data = data.dropna(subset=["MILES"])

# Fill missing categorical values
data["PURPOSE"] = data["PURPOSE"].fillna("Not Specified")
data["CATEGORY"] = data["CATEGORY"].fillna("Unknown")

# Create trip duration in minutes
data["DURATION_MIN"] = (
    (data["END_DATE"] - data["START_DATE"]).dt.total_seconds() / 60
)

# Remove invalid trips
data = data[data["DURATION_MIN"] > 0]
data = data[data["MILES"] > 0]

# Create analysis features
data["HOUR"] = data["START_DATE"].dt.hour
data["DAY"] = data["START_DATE"].dt.day_name()
data["MONTH"] = data["START_DATE"].dt.month_name()

print("\nAfter cleaning:", data.shape)

# Save cleaned data
data.to_csv("uber_clean.csv", index=False)


## Step 3 — Basic Statistics

In [ ]:
# Basic statistics
print("\n--- Basic Stats ---")
print(data["MILES"].describe())
print(data["CATEGORY"].value_counts())
print(data["PURPOSE"].value_counts())

total_miles = data["MILES"].sum()
avg_miles = data["MILES"].mean()

print("Total miles:", total_miles)
print("Average miles per trip:", avg_miles)


## Step 4 — Visualization: Rides by Hour

In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(10, 5))
data["HOUR"].value_counts().sort_index().plot(kind="bar")
plt.title("Rides by Hour of the Day")
plt.xlabel("Hour")
plt.ylabel("Number of Rides")
plt.tight_layout()
plt.savefig("rides_by_hour.png")
plt.show()


## Visualization: Rides by Day

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

plt.figure(figsize=(9, 5))
data["DAY"].value_counts().reindex(day_order).plot(kind="bar")
plt.title("Rides by Day of Week")
plt.xlabel("Day")
plt.ylabel("Number of Rides")
plt.tight_layout()
plt.savefig("rides_by_day.png")
plt.show()


## Visualization: Peak Hours Heatmap

In [ ]:
pivot_table = data.pivot_table(
    index="DAY",
    columns="HOUR",
    values="MILES",
    aggfunc="count",
    fill_value=0
)
pivot_table = pivot_table.reindex(day_order)

plt.figure(figsize=(12, 5))
sns.heatmap(pivot_table, cmap="YlOrRd")
plt.title("Peak Hours Heatmap (Day vs Hour)")
plt.tight_layout()
plt.savefig("peak_hours_heatmap.png")
plt.show()


## Visualization: Monthly Trend

In [ ]:
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

plt.figure(figsize=(10, 5))
data["MONTH"].value_counts().reindex(month_order).plot(
    kind="line", marker="o"
)
plt.title("Monthly Ride Trend")
plt.xlabel("Month")
plt.ylabel("Number of Rides")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("monthly_trend.png")
plt.show()


## Visualization: Purpose Breakdown

In [ ]:
plt.figure(figsize=(9, 5))
data["PURPOSE"].value_counts().plot(kind="barh")
plt.title("Rides by Purpose")
plt.xlabel("Number of Rides")
plt.tight_layout()
plt.savefig("purpose_breakdown.png")
plt.show()


## Visualization: Business vs Personal

In [ ]:
plt.figure(figsize=(6, 6))
data["CATEGORY"].value_counts().plot(kind="pie", autopct="%1.1f%%")
plt.title("Business vs Personal Rides")
plt.ylabel("")
plt.tight_layout()
plt.savefig("category_split.png")
plt.show()


## Visualization: Distance Distribution

In [ ]:
# The supplied README notes that the dataset has no fare/price column,
# so trip miles are used as the distance/fare proxy.
plt.figure(figsize=(9, 5))
sns.histplot(data[data["MILES"] < 50]["MILES"], bins=30, kde=True)
plt.title("Distribution of Trip Distance (miles)")
plt.xlabel("Miles")
plt.tight_layout()
plt.savefig("miles_distribution.png")
plt.show()


## Visualization: Top Pickup Locations

In [ ]:
plt.figure(figsize=(9, 5))
data["START"].value_counts().head(10).plot(kind="barh")
plt.title("Top 10 Pickup Locations")
plt.xlabel("Number of Rides")
plt.tight_layout()
plt.savefig("top_locations.png")
plt.show()


## Step 5 — Random Forest Demand Model

In [ ]:
# Group rides into date + hour buckets
data["DATE"] = data["START_DATE"].dt.date

demand = (
    data.groupby(["DATE", "HOUR"])
    .size()
    .reset_index(name="RIDE_COUNT")
)

demand["DATE"] = pd.to_datetime(demand["DATE"])
demand["DAY_NUM"] = demand["DATE"].dt.dayofweek
demand["MONTH_NUM"] = demand["DATE"].dt.month

X = demand[["HOUR", "DAY_NUM", "MONTH_NUM"]]
y = demand["RIDE_COUNT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=150,
    random_state=42
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("\n--- Model Results ---")
print("MAE:", mean_absolute_error(y_test, pred))
print("R2 Score:", r2_score(y_test, pred))


## Model Feature Importance

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nFeature Importance:")
print(importance)

plt.figure(figsize=(7, 4))
importance.plot(kind="bar")
plt.title("Feature Importance for Demand Prediction")
plt.tight_layout()
plt.savefig("feature_importance.png")
plt.show()


## Key Insights

# Key insights from the supplied README
print("Total rides after cleaning: approximately 1,151")
print("Total miles: approximately 12,130")
print("Average trip: approximately 10.5 miles")
print("Most rides are Business category (1074 vs 77 Personal)")
print("Busiest hour: around 3 PM")
print("Busiest day: Friday")
print("Busiest month: December")
print("Most common filled purpose: Meeting")


## Conclusion and Limitations

## Conclusion

The analysis covers:
- Data loading and cleaning
- Date/time feature extraction
- Ride volume analysis by hour, day and month
- Trip-purpose and category analysis
- Distance distribution
- Pickup-location analysis
- Random Forest ride-demand prediction

### Model limitation

The supplied README reports a low R² (negative on some runs) because this is one person's ride history and most date/hour combinations have very few rides. A larger multi-user dataset would provide more repeated patterns and make demand forecasting more useful.

The dataset also does not contain a fare/price column, so miles are used as a proxy for distance/fare-related analysis.
